# ML Lab 12 
# Muhammad Sikandar Hussain 502808

### importing libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler


### 1. Load the dataset

In [7]:
path = "automobile/imports-85.data"
headers = ["symboling", "normalized-losses", "make", "fuel-type", "aspiration",
           "num-of-doors", "body-style", "drive-wheels", "engine-location",
           "wheel-base", "length", "width", "height", "curb-weight",
           "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
           "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
           "city-mpg", "highway-mpg", "price"]
df = pd.read_csv(path, names=headers)
print(df.shape)
df.head()

(205, 26)


,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450


### Task 2: Perform data cleaning (Missing values, duplicates, outliers)

In [12]:
#replace missing with NAN
df = df.replace('?', np.nan,inplace=True)

numeric_cols = ["normalized-losses", "bore", "stroke", "horsepower", "peak-rpm", "price"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
df.head(2)


,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0


In [14]:
# Drop rows where our target variable 'price' is missing
df.dropna(subset=['price'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head(2)

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0


In [16]:
# Handle Missing Values: Impute continuous variables with the mean and categorical with the mode
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)
df['num-of-doors'].fillna(df['num-of-doors'].mode()[0], inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_12044\2756286810.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].mean(), inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_12044\2756286810.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

0       two
1       two
2       two
3      four
4      four
       ... 
196    four
197    four
198    four
199    four
200    four
Name: num-of-doors, Length: 201, dtype: str

In [17]:
# Handle Outliers: Cap extreme outliers in 'engine-size' and 'horsepower' using the IQR method
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])

cap_outliers(df, 'engine-size')
cap_outliers(df, 'horsepower')

print("Missing values after cleaning:\n", df.isnull().sum().sum())

Missing values after cleaning:
 51


### Task 3: Handle Base Model (Before Feature Engineering) 

In [18]:
base_df = df.select_dtypes(include=[np.number])
X_base = base_df.drop('price', axis=1)
y_base = base_df['price']

X_train_base, X_test_base, y_train, y_test = train_test_split(X_base, y_base, test_size=0.2, random_state=42)

# Train a baseline Random Forest Regressor
rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_train_base, y_train)
y_pred_base = rf_base.predict(X_test_base)

print("--- Base Model Evaluation (Numeric Features Only) ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_base)):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred_base):.4f}")

--- Base Model Evaluation (Numeric Features Only) ---
RMSE: 2754.22
R2 Score: 0.9380


### Task 4 & 5: Feature Selection, Scaling, and Transformation

In [19]:
df_fe = df.copy()

# 5. Feature Transformation: Create a new feature 'avg-mpg' blending city and highway mpg
df_fe['avg-mpg'] = (df_fe['city-mpg'] + df_fe['highway-mpg']) / 2

# Map word-based numbers to actual integers
num_map = {'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'eight': 8, 'twelve': 12}
df_fe['num-of-doors'] = df_fe['num-of-doors'].map(num_map)
df_fe['num-of-cylinders'] = df_fe['num-of-cylinders'].map(num_map)

# 3. Feature Selection: Drop irrelevant or redundant columns
# We drop 'city-mpg' and 'highway-mpg' since we created 'avg-mpg'. 
# 'symboling' and 'normalized-losses' have weak correlation to price and high NA originally.
columns_to_drop = ['city-mpg', 'highway-mpg', 'symboling', 'normalized-losses']
df_fe.drop(columns_to_drop, axis=1, inplace=True)

# One-hot encode the remaining categorical variables
categorical_cols = df_fe.select_dtypes(include=['object']).columns
df_fe = pd.get_dummies(df_fe, columns=categorical_cols, drop_first=True)

# Prepare engineered data
X_fe = df_fe.drop('price', axis=1)
y_fe = df_fe['price']

# Split engineered data
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, test_size=0.2, random_state=42)

# 4. Feature Scaling: Scale the numeric features using StandardScaler
# (Scaling is highly useful for linear models and fast convergence in others)
scaler = StandardScaler()
X_train_fe_scaled = scaler.fit_transform(X_train_fe)
X_test_fe_scaled = scaler.transform(X_test_fe)

C:\Users\DELL\AppData\Local\Temp\ipykernel_12044\1423757299.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_fe.select_dtypes(include=['object']).columns


### Task 6: Evaluate the performance after Feature Engineering

In [22]:
from sklearn.impute import SimpleImputer

# Impute missing values after scaling
imputer = SimpleImputer(strategy='median')
X_train_fe_ready = imputer.fit_transform(X_train_fe_scaled)
X_test_fe_ready = imputer.transform(X_test_fe_scaled)

# Evaluate Random Forest Regressor on Engineered Data
rf_fe = RandomForestRegressor(random_state=42)
rf_fe.fit(X_train_fe_ready, y_train_fe)
y_pred_rf_fe = rf_fe.predict(X_test_fe_ready)

print("--- Engineered Model Evaluation (Random Forest) ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_fe, y_pred_rf_fe)):.2f}")
print(f"R2 Score: {r2_score(y_test_fe, y_pred_rf_fe):.4f}")

# Bonus: Compare with Linear Regression
lr = LinearRegression()
lr.fit(X_train_fe_ready, y_train_fe)
y_pred_lr = lr.predict(X_test_fe_ready)

print("\n--- Engineered Model Evaluation (Linear Regression) ---")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_fe, y_pred_lr)):.2f}")
print(f"R2 Score: {r2_score(y_test_fe, y_pred_lr):.4f}")

--- Engineered Model Evaluation (Random Forest) ---
RMSE: 2675.94
R2 Score: 0.9415

--- Engineered Model Evaluation (Linear Regression) ---
RMSE: 3023.46
R2 Score: 0.9253
